# EDA

## Basic data exploration

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/raw/revenues_per_day.csv")
display(df.head())

,id,date,title,revenue,theaters,distributor
0,8b19ad43-3a7e-b14b-49e9-1f7a0eb1568e,2004-09-20,Sky Captain and the World of Tomorrow,925482,3170.0,Paramount Pictures
1,481fc700-fcdd-1919-c53c-09fcd423a596,2004-09-20,Resident Evil: Apocalypse,643680,3284.0,Screen Gems
2,06719cc2-c05e-8c0b-56b6-b8b13e056509,2004-09-20,Mr 3000,425375,2736.0,Walt Disney Studios Motion Pictures
3,3be8e4a8-2716-00be-80be-eb342ba3cbe7,2004-09-20,Wimbledon,416970,2034.0,Universal Pictures
4,c5f26b59-6a64-ae8e-8f58-7cefb09eadfe,2004-09-20,Cellular,412000,2749.0,New Line Cinema


In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 337818 entries, 0 to 337817
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   id           337818 non-null  str    
 1   date         337818 non-null  str    
 2   title        337818 non-null  str    
 3   revenue      337818 non-null  int64  
 4   theaters     337657 non-null  float64
 5   distributor  337593 non-null  str    
dtypes: float64(1), int64(1), str(4)
memory usage: 41.2 MB


In [6]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("---")
print("Data types:")
print(df.dtypes)
print("---")
print("Missing values:")
print(df.isna().sum())
print("---")
print("Unique titles:", df["title"].nunique())
print("Unique theaters:", df["theaters"].nunique())
print("Unique distributors:", df["distributor"].nunique())
print("---")
df["date"] = pd.to_datetime(df["date"])
print("Date range:", df["date"].min() , "-", df["date"].max())
print("Full duplicates:", df.duplicated().sum(), )
print("Title duplicates", df.duplicated(subset=["title"]).sum())
print("Title+date duplicates", df.duplicated(subset=["title", "date"]).sum() )



Rows: 337818
Columns: ['id', 'date', 'title', 'revenue', 'theaters', 'distributor']
---
Data types:
id                 str
date               str
title              str
revenue          int64
theaters       float64
distributor        str
dtype: object
---
Missing values:
id               0
date             0
title            0
revenue          0
theaters       161
distributor    225
dtype: int64
---
Unique titles: 6545
Unique theaters: 4114
Unique distributors: 362
---
Date range: 2000-01-01 00:00:00 - 2023-03-06 00:00:00
Full duplicates: 0
Title duplicates 331273
Title+date duplicates 0


In [7]:
print("Unique vals:")
print(df.nunique())

Unique vals:
id             337818
date             8455
title            6545
revenue        179113
theaters         4114
distributor       362
dtype: int64


In [8]:
for col in df.columns:
    print(f"Column '{col}': {df[col].duplicated().sum()} duplicates")

Column 'id': 0 duplicates
Column 'date': 329363 duplicates
Column 'title': 331273 duplicates
Column 'revenue': 158705 duplicates
Column 'theaters': 333703 duplicates
Column 'distributor': 337455 duplicates


## Summary

Records amount: 337818

Date range: 2000-01-01 00:00:00 - 2023-03-06 00:00:00

Duplicates: 0

Full duplicates: 0

Title+date duplicates: 0

Column | Type | Missing values | Unique values
--- | --- | --- | ---
id | str | 0 | 337818
date | datetime64[us] | 0 | 8455
title | str | 0 | 6545
revenue | int64 | 0  | 179113
theaters | float64 | 161 | 4114
distributor | str | 225 | 362



## Assumption I.

Date represents box-office revenue rather than the movie release date

In [9]:
example_title = (
    df["title"]
    .value_counts()
    .index[0]
)

display(
    df[df["title"] == example_title]
    [["date", "title", "revenue", "theaters"]]
    .sort_values("date")
    .head(20)
)

,date,title,revenue,theaters
244189,2006-03-03,Deep Sea,139786,43.0
119364,2006-03-04,Deep Sea,326478,43.0
53144,2006-03-05,Deep Sea,233949,43.0
298342,2006-03-06,Deep Sea,58578,43.0
93232,2006-03-07,Deep Sea,53516,43.0
206466,2006-03-08,Deep Sea,62884,43.0
157221,2006-03-09,Deep Sea,76488,43.0
267438,2006-03-10,Deep Sea,140913,44.0
72967,2006-03-11,Deep Sea,323894,44.0
143668,2006-03-12,Deep Sea,247676,44.0


In [10]:
days_per_title = (
    df.groupby("title")["date"]
    .nunique()
    .sort_values(ascending=False)
)

print(days_per_title.describe())

count    6545.000000
mean       51.614668
std        49.308114
min         1.000000
25%        13.000000
50%        42.000000
75%        77.000000
max      1114.000000
Name: date, dtype: float64


In [11]:
print(days_per_title.head(10))

title
Deep Sea                                 1114
Born to Be Wild                           822
Under the Sea 3D                          695
Island of Lemurs: Madagascar              533
Hubble 3D                                 502
To the Arctic 3D                          441
NASCAR: The IMAX Experience               371
Winter's Bone                             300
Gladiator                                 248
The Lord of the Rings: The Two Towers     246
Name: date, dtype: int64


### Conclusion I.
Date represents box-office revenue rather than the movie release date.

### Proceeding
**We can't query the database for `{title, date}`!**


In [12]:
df["revenue_year"] = df["date"].dt.year

years_per_title = (
    df.groupby("title")["revenue_year"]
    .nunique()
)

print(years_per_title.value_counts().sort_index())

revenue_year
1    5397
2    1113
3      23
4       6
5       3
6       2
9       1
Name: count, dtype: int64



## Assumption II.
Some movies might have suspicious gap between dates

In [13]:
sorted_df = df.sort_values(["title", "date"]).copy()

sorted_df["previous_date"] = (
    sorted_df.groupby("title")["date"].shift()
)

sorted_df["gap_days"] = (
    sorted_df["date"] - sorted_df["previous_date"]
).dt.days

title_gaps = (
    sorted_df.groupby("title")["gap_days"]
    .max()
    .sort_values(ascending=False)
)

ambiguous_titles = title_gaps[title_gaps > 365]

print("Titles with a gap > 365 days:", len(ambiguous_titles))
print(ambiguous_titles.head(20))

Titles with a gap > 365 days: 79
title
Charlie's Angels       6886.0
The Kid                6763.0
Pinocchio              6567.0
The Lost City          5804.0
The Grudge             5493.0
Confetti               5438.0
Hellboy                5437.0
City of Ghosts         5184.0
The Cave               5131.0
Infamous               4922.0
Iris                   4844.0
Serenity               4817.0
Stardust               4780.0
Venom                  4765.0
Black Christmas        4684.0
War                    4381.0
Beautiful Creatures    4291.0
Amazing Grace          4250.0
Nobody                 4194.0
Criminal               4187.0
Name: gap_days, dtype: float64


### Conclusion II.
79 titles contain a gap longer than 365 days. 

These titles may represent re-releases or different movies with the same name. 

### Proceeding

They are flagged as `needs_review` in the pipeline instead of being treated as confirmed OMDb matches.